# Bobot ensemble dengan metaheuristik: 7 cara mencari bobot, dinilai jujur
**Kapan dipakai:** sudah ada ≥ 3 file OOF dan metrik lomba tidak halus (F1 makro, akurasi, QWK),
sehingga optimasi gradien (Nelder-Mead/SLSQP di `ensemble_oof`) sering macet. Metaheuristik
tidak butuh turunan.

| cara | ide | catatan |
|---|---|---|
| tunggal | model terbaik saja | patokan |
| rata | bobot sama | sering sulit dikalahkan |
| hill | Caruana: tambah model terbaik berulang (boleh berulang) | ada di `ensemble_oof` |
| hill negatif | gaya Chris Deotte: bobot boleh negatif | hanya metrik argmax/peringkat |
| annealing | simulated annealing: kadang menerima langkah lebih buruk | lolos dari puncak lokal |
| evolusi diferensial | `scipy.optimize.differential_evolution` | populasi, tanpa turunan |
| genetik | seleksi, persilangan, mutasi, elitisme | populasi, tanpa turunan |
| optuna | TPE (Bayesian) atas bobot | bawaan repo |

**Skor jujur:** bobot dicari di 4/5 baris OOF, dinilai di 1/5 sisanya (lipatan luar).
Skor OOF penuh selalu optimis karena bobot dicari dan dinilai di baris yang sama.

In [ ]:
import os
import time
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/bobot_ensemble_metaheuristik"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/classification"    # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/emot_train.csv"                  # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/emot_test.csv"                    # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/emot_test_labels.csv"              # GANTI saat lomba (kunci, kalau ada)
ID_COL = "id"                                # GANTI saat lomba
TARGET = "label"                             # GANTI saat lomba
MODELS = ["sgd_kata", "sgd_huruf", "cnb_kata", "ridge_kata", "knn_svd"]   # GANTI: nama OOF
N_FOLDS = 5
MIN_GAIN = 0.002             # cara rumit harus menang ≥ ini atas cara yang lebih sederhana
HILL_STEPS = 30
NEG_GRID = np.r_[np.arange(-0.5, 0, 0.05), np.arange(0.05, 0.55, 0.05)]
ANNEAL_STEPS = 400
ANNEAL_START = 0.01          # suhu awal ≈ besar selisih skor yang masih mau diterima
ANNEAL_COOLING = 0.99
DE_ITER = 20
GA_POP = 30
GA_GENERATIONS = 25
GA_MUTATION = 0.08
OPTUNA_TRIALS = 200
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca target + lipatan

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, y))
print("train:", train.shape, "| test:", test.shape, "| kelas:", list(CLASSES))

## OOF demo (HAPUS saat lomba)
Lima model cepat yang sengaja beragam, ditulis ke `OUTPUT_FOLDER` (folder OOF asli tidak
dikotori). Ridge tidak punya peluang: skornya diubah dengan softmax.

In [ ]:
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import RidgeClassifier, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

text = pd.concat([train["text"], test["text"]], ignore_index=True).fillna("")
counts = CountVectorizer(ngram_range=(1, 2), min_df=2).fit_transform(text)
word = TfidfTransformer(sublinear_tf=True).fit_transform(counts)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True,
                       max_features=100000).fit_transform(text)
both = hstack([word, char]).tocsr()
svd = normalize(TruncatedSVD(128, random_state=SEED).fit_transform(both))
OOF_FOLDER = OUTPUT_FOLDER
print("matriks demo:", word.shape, char.shape, svd.shape)

In [ ]:
def softmax(scores):
    shifted = np.exp(scores - scores.max(axis=1, keepdims=True))
    return shifted / shifted.sum(axis=1, keepdims=True)


def member_proba(fitted, rows):
    if hasattr(fitted, "predict_proba"):
        return fitted.predict_proba(rows)
    return softmax(fitted.decision_function(rows))


def demo_member(matrix, model, name):
    oof = np.zeros((len(train), len(CLASSES)))
    test_pred = np.zeros((len(test), len(CLASSES)))
    for idx_fit, idx_val in folds:
        model.fit(matrix[idx_fit], y[idx_fit])
        oof[idx_val] = member_proba(model, matrix[idx_val])
        test_pred += member_proba(model, matrix[len(train):]) / N_FOLDS
    np.save(f"{OOF_FOLDER}/oof_{name}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{name}.npy", test_pred)


print("uji softmax:", softmax(np.array([[1.0, 1.0]])))

In [ ]:
demo_member(word, SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED), "sgd_kata")
demo_member(char, SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED), "sgd_huruf")
demo_member(counts, ComplementNB(alpha=0.3), "cnb_kata")
demo_member(word, RidgeClassifier(alpha=1.0), "ridge_kata")
demo_member(svd, KNeighborsClassifier(25, weights="distance"), "knn_svd")
print("OOF demo ditulis")

## Muat OOF semua model
`stack` berbentuk (model, baris, kelas). Kolom kelas = `np.unique(train[TARGET])`.

In [ ]:
oof_list = []
test_list = []
for name in MODELS:
    oof_list.append(np.load(f"{OOF_FOLDER}/oof_{name}.npy").reshape(len(train), -1))
    test_list.append(np.load(f"{OOF_FOLDER}/test_{name}.npy").reshape(len(test), -1))
stack = np.stack(oof_list)
test_stack = np.stack(test_list)
for number, name in enumerate(MODELS):
    print(f"{name:12s} | OOF {metric(y, stack[number]):.4f}")

## Alat bersama
Semua cara memakai antarmuka sama: `cara(members, truth) → bobot` (panjang = jumlah model).

In [ ]:
def blend(weights, members):
    return np.tensordot(weights, members, axes=1)


def fitness(weights, members, truth):
    return metric(truth, blend(weights, members))


def to_simplex(raw):
    weights = np.clip(raw, 0, None) + 1e-12
    return weights / weights.sum()


def single_scores(members, truth):
    scores = np.zeros(len(members))
    for number in range(len(members)):
        scores[number] = metric(truth, members[number])
    return scores


print("uji simplex:", to_simplex(np.array([2.0, -1.0, 2.0])))

## Cara 1–3: tunggal, rata, hill climbing Caruana

In [ ]:
def single_weights(members, truth):
    weights = np.zeros(len(members))
    weights[int(np.argmax(single_scores(members, truth)))] = 1.0
    return weights


def mean_weights(members, truth):
    return np.ones(len(members)) / len(members)


def hill_weights(members, truth):
    counts_used = single_weights(members, truth)
    best = fitness(counts_used, members, truth)
    for _ in range(HILL_STEPS):
        trials = np.zeros(len(members))
        for number in range(len(members)):
            trial = counts_used.copy()
            trial[number] += 1
            trials[number] = fitness(trial / trial.sum(), members, truth)
        if trials.max() <= best:
            break
        best = trials.max()
        counts_used[int(np.argmax(trials))] += 1
    return counts_used / counts_used.sum()


print("uji hill:", hill_weights(stack[:, :500], y[:500]).round(2))

## Cara 4: hill climbing bobot negatif (gaya Chris Deotte)
Tiap langkah: `bobot = (1 − w) × bobot + w × model`, dengan w ∈ `NEG_GRID` (boleh negatif =
"kurangi" model yang kesalahannya mirip). Jangan dipakai untuk logloss (peluang bisa negatif).

In [ ]:
def hill_negative_weights(members, truth):
    weights = single_weights(members, truth)
    best = fitness(weights, members, truth)
    for _ in range(HILL_STEPS):
        found = None
        for number in range(len(members)):
            for step in NEG_GRID:
                trial = weights * (1 - step)
                trial[number] += step
                value = fitness(trial, members, truth)
                if value > best:
                    best, found = value, trial
        if found is None:
            break
        weights = found
    return weights


print("uji hill negatif:", hill_negative_weights(stack[:, :500], y[:500]).round(2))

## Cara 5: simulated annealing
Satu bobot diganggu acak tiap langkah. Langkah lebih buruk tetap diterima dengan peluang
`exp(selisih / suhu)`; suhu turun pelan (`ANNEAL_COOLING`), jadi di akhir hanya naik.

In [ ]:
def annealing_weights(members, truth):
    rng = np.random.default_rng(SEED)
    current = mean_weights(members, truth)
    current_value = fitness(current, members, truth)
    best, best_value = current, current_value
    temperature = ANNEAL_START
    for _ in range(ANNEAL_STEPS):
        trial = current.copy()
        number = rng.integers(len(members))
        trial[number] = max(0.0, trial[number] + rng.normal(0, 0.15))
        trial = to_simplex(trial)
        value = fitness(trial, members, truth)
        if value > current_value or rng.random() < np.exp((value - current_value) / temperature):
            current, current_value = trial, value
        if current_value > best_value:
            best, best_value = current, current_value
        temperature *= ANNEAL_COOLING
    return best


print("uji annealing:", annealing_weights(stack[:, :500], y[:500]).round(2))

## Cara 6–7: evolusi diferensial (scipy) + algoritma genetik

In [ ]:
def negative_fitness(raw, members, truth):
    return -fitness(to_simplex(raw), members, truth)


def evolution_weights(members, truth):
    bounds = [(0.0, 1.0)] * len(members)
    found = differential_evolution(negative_fitness, bounds, args=(members, truth),
                                   maxiter=DE_ITER, popsize=10, seed=SEED, polish=False, tol=0)
    return to_simplex(found.x)


print("uji evolusi:", evolution_weights(stack[:, :500], y[:500]).round(2))

In [ ]:
def genetic_weights(members, truth):
    rng = np.random.default_rng(SEED)
    population = rng.dirichlet(np.ones(len(members)), GA_POP)
    for _ in range(GA_GENERATIONS):
        values = np.zeros(GA_POP)
        for number in range(GA_POP):
            values[number] = fitness(population[number], members, truth)
        order = np.argsort(values)[::-1]
        parents = population[order[: GA_POP // 2]]
        children = [population[order[0]]]
        while len(children) < GA_POP:
            pair = parents[rng.integers(len(parents), size=2)]
            mix = rng.random()
            noise = rng.normal(0, GA_MUTATION, len(members))
            children.append(to_simplex(mix * pair[0] + (1 - mix) * pair[1] + noise))
        population = np.array(children)
    return population[0]


print("uji genetik:", genetic_weights(stack[:, :500], y[:500]).round(2))

## Cara 8: Optuna TPE

In [ ]:
def optuna_objective(trial, members, truth):
    raw = np.zeros(len(members))
    for number in range(len(members)):
        raw[number] = trial.suggest_float(f"w{number}", 0.0, 1.0)
    return fitness(to_simplex(raw), members, truth)


def optuna_weights(members, truth):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(partial(optuna_objective, members=members, truth=truth),
                   n_trials=OPTUNA_TRIALS)
    raw = np.zeros(len(members))
    for number in range(len(members)):
        raw[number] = study.best_params[f"w{number}"]
    return to_simplex(raw)


METHODS = {"tunggal": single_weights, "rata": mean_weights, "hill": hill_weights,
           "hill negatif": hill_negative_weights, "annealing": annealing_weights,
           "evolusi diferensial": evolution_weights, "genetik": genetic_weights,
           "optuna": optuna_weights}
print("cara dibandingkan:", list(METHODS))

## Skor jujur semua cara (lipatan luar)

In [ ]:
outer = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1).split(train, y))
rows = []
FOUND = {}
for name, search in METHODS.items():
    start = time.time()
    honest = np.zeros_like(stack[0])
    for idx_fit, idx_val in outer:
        weights = search(stack[:, idx_fit], y[idx_fit])
        honest[idx_val] = blend(weights, stack[:, idx_val])
    FOUND[name] = search(stack, y)
    rows.append({"cara": name, "skor OOF (optimis)": round(fitness(FOUND[name], stack, y), 4),
                 "skor jujur": round(metric(y, honest), 4), "detik": round(time.time() - start)})
    print(rows[-1])
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_cara.csv", index=False)

## Putusan + bobot
Pilih cara PALING SEDERHANA (urutan `METHODS`) yang skor jujurnya tidak kalah dari yang terbaik
lebih dari `MIN_GAIN`.

In [ ]:
honest_scores = pd.Series(table["skor jujur"].to_numpy(), index=table["cara"])
best_value = honest_scores.max()
CHOSEN = None
for name in METHODS:
    if CHOSEN is None and honest_scores[name] >= best_value - MIN_GAIN:
        CHOSEN = name
weights_table = pd.DataFrame(FOUND, index=MODELS).round(3)
weights_table.to_csv(f"{OUTPUT_FOLDER}/bobot_semua_cara.csv")
print("dipilih:", CHOSEN, "| skor jujur", honest_scores[CHOSEN], "| terbaik", best_value)
print(weights_table.to_string())

## Gambar untuk slide + tebakan test

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(table["cara"], table["skor jujur"], color="#0072B2")
axes[0].set_xlim(table["skor jujur"].min() - 0.01, table["skor jujur"].max() + 0.005)
axes[0].set_xlabel("skor jujur (lipatan luar)")
axes[0].invert_yaxis()
axes[1].barh(MODELS, FOUND[CHOSEN], color="#E69F00")
axes[1].set_xlabel(f"bobot cara '{CHOSEN}'")
axes[1].invert_yaxis()
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/banding_cara.png", dpi=200, bbox_inches="tight")
plt.show()
final_test = blend(FOUND[CHOSEN], test_stack)
np.save(f"{OUTPUT_FOLDER}/tebakan_test_proba.npy", final_test)
guess = pd.DataFrame()
guess[ID_COL] = test[ID_COL]
guess[TARGET] = CLASSES[final_test.argmax(axis=1)]
guess.to_csv(f"{OUTPUT_FOLDER}/tebakan_test.csv", index=False)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth_test = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
for name in METHODS:
    print(f"{name:20s} | test {fitness(FOUND[name], test_stack, truth_test):.4f}")